# Week 80, Day 2: Project: database and models

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w080d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('helpdesk_domain.py', 'w', encoding='utf-8') as f:
    f.write('"""The helpdesk\'s rules, with no web or database code (Week 80, Day 1).\n\nEverything here is plain data and pure functions, so it can be tested on its own and used by the\ndatabase layer, the web layer and the tests alike.\n"""\nimport re\nfrom http import HTTPStatus\n\nROLES = ("customer", "agent", "admin")\nPRIORITIES = ("low", "normal", "high", "urgent")\nSTATUSES = ("open", "in_progress", "waiting", "resolved", "closed")\nSTAFF = {"agent", "admin"}\n\n# status -> {new status: roles that may make the change}\nTRANSITIONS = {\n    "open": {"in_progress": {"agent", "admin"}, "closed": {"customer", "admin"}},\n    "in_progress": {"waiting": {"agent", "admin"}, "resolved": {"agent", "admin"}},\n    "waiting": {"in_progress": {"customer", "agent", "admin"}},\n    "resolved": {"closed": {"customer", "admin"}, "in_progress": {"customer", "agent", "admin"}},\n    "closed": {},\n}\n\n# action -> (roles always allowed, roles allowed only on their own tickets)\nPERMISSIONS = {\n    "create_ticket": ({"customer"}, set()),\n    "view_ticket": ({"agent", "admin"}, {"customer"}),\n    "list_all_tickets": ({"agent", "admin"}, set()),\n    "edit_ticket": ({"admin"}, {"customer"}),\n    "comment": ({"agent", "admin"}, {"customer"}),\n    "internal_note": ({"agent", "admin"}, set()),\n    "change_priority": ({"agent", "admin"}, set()),\n    "assign": ({"agent", "admin"}, set()),\n    "delete_ticket": ({"admin"}, set()),\n    "view_stats": ({"agent", "admin"}, set()),\n}\n\n\nclass ValidationError(Exception):\n    """Raised with a dict of field -> message."""\n\n    def __init__(self, errors):\n        super().__init__("; ".join(f"{k}: {v}" for k, v in sorted(errors.items())))\n        self.errors = errors\n\n\ndef validate_ticket(data, partial=False):\n    """Check a ticket body from a client. partial=True for PATCH: only the fields present are checked."""\n    if not isinstance(data, dict):\n        raise ValidationError({"_": "expected a JSON object"})\n    errors, clean = {}, {}\n    unknown = sorted(set(data) - {"title", "body", "priority"})\n    if unknown:\n        errors["_"] = f"unexpected field(s): {\', \'.join(unknown)}"\n    for field in ("title", "body"):\n        if field not in data:\n            if not partial:\n                errors[field] = "required"\n            continue\n        value = data[field]\n        limit = (3, 120) if field == "title" else (1, 5000)\n        if not isinstance(value, str) or not limit[0] <= len(value.strip()) <= limit[1]:\n            errors[field] = f"{limit[0]} to {limit[1]} characters"\n        else:\n            clean[field] = value.strip()\n    if "priority" in data:\n        if data["priority"] not in PRIORITIES:\n            errors["priority"] = f"one of {\', \'.join(PRIORITIES)}"\n        else:\n            clean["priority"] = data["priority"]\n    elif not partial:\n        clean["priority"] = "normal"\n    if partial and not data:\n        errors["_"] = "nothing to change"\n    if errors:\n        raise ValidationError(errors)\n    return clean\n\n\ndef can_transition(current, new, role):\n    """True if someone with this role may move a ticket from `current` to `new`."""\n    return role in TRANSITIONS.get(current, {}).get(new, set())\n\n\ndef allowed(role, action, is_owner=False):\n    """True if the role may do the action (is_owner: the ticket is the user\'s own)."""\n    always, own = PERMISSIONS[action]\n    return role in always or (is_owner and role in own)\n\n\ndef problem(status, detail, title=None, **extra):\n    """An RFC 9457 problem-details response: (body, status, headers)."""\n    body = {"type": "about:blank", "title": title or HTTPStatus(status).phrase, "status": status, "detail": detail}\n    body.update(extra)\n    return body, status, {"Content-Type": "application/problem+json"}\n\n\nROUTES = [\n    ("POST", "/users", "sign up as a customer", 201),\n    ("POST", "/tokens", "log in and get a token", 201),\n    ("GET", "/me", "who am I", 200),\n    ("GET", "/tickets", "list tickets (customers see their own)", 200),\n    ("POST", "/tickets", "open a ticket", 201),\n    ("GET", "/tickets/{ticket_id}", "read a ticket", 200),\n    ("PATCH", "/tickets/{ticket_id}", "change a ticket\'s title, body or priority", 200),\n    ("DELETE", "/tickets/{ticket_id}", "delete a ticket (admins)", 204),\n    ("POST", "/tickets/{ticket_id}/transitions", "change a ticket\'s status", 200),\n    ("PUT", "/tickets/{ticket_id}/assignee", "assign a ticket to an agent", 200),\n    ("GET", "/tickets/{ticket_id}/comments", "list a ticket\'s comments", 200),\n    ("POST", "/tickets/{ticket_id}/comments", "add a comment or internal note", 201),\n    ("GET", "/stats", "ticket counts (staff)", 200),\n    ("GET", "/healthz", "is the process alive", 200),\n    ("GET", "/readyz", "can it serve requests", 200),\n]\n\n\ndef docs_table(routes):\n    """The routes as a Markdown table, for the README."""\n    lines = ["| Method | Path | What it does | Success |", "|---|---|---|---|"]\n    lines += [f"| {m} | `{p}` | {what} | {code} {HTTPStatus(code).phrase} |" for m, p, what, code in routes]\n    return "\\n".join(lines)\n')

**Predict before you run:** what will this print?



In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, status TEXT NOT NULL CHECK (status IN ('open', 'closed')))")
conn.execute("INSERT INTO t (status) VALUES ('open')")
for bad in ("lost", None):
    try:
        conn.execute("INSERT INTO t (status) VALUES (?)", (bad,))
    except sqlite3.IntegrityError as e:
        print(type(e).__name__, str(e).split(":")[0])
print(conn.execute("SELECT id, status FROM t").fetchall())

### Exercise: Step 6: migrations

Write `migrate(conn, migrations=MIGRATIONS)`. Create the table `schema_migrations (version INTEGER PRIMARY KEY, name TEXT NOT NULL)` if it doesn't exist, and commit. If the versions in `migrations` aren't exactly `1, 2, 3...` in order, raise `ValueError("migrations must be numbered 1, 2, 3... with no gaps")`. Read the current version (the highest recorded, or 0); if it's higher than the number of migrations, raise `RuntimeError(f"the database is at version N, newer than this code")`. Then apply each pending migration in order, each in its own transaction: `BEGIN`, every non-blank statement of its SQL split at `;`, then the insert into `schema_migrations`, then commit; on any error, roll back and re-raise. Return the list of versions applied.

In [ ]:
import sqlite3

MIGRATIONS = [
    (1, "users", """
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            password TEXT NOT NULL,
            role TEXT NOT NULL CHECK (role IN ('customer', 'agent', 'admin')),
            created_at TEXT NOT NULL
        )"""),
    (2, "tickets", """
        CREATE TABLE tickets (
            id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES users (id),
            assignee_id INTEGER REFERENCES users (id),
            title TEXT NOT NULL,
            body TEXT NOT NULL,
            priority TEXT NOT NULL DEFAULT 'normal' CHECK (priority IN ('low', 'normal', 'high', 'urgent')),
            status TEXT NOT NULL DEFAULT 'open'
                CHECK (status IN ('open', 'in_progress', 'waiting', 'resolved', 'closed')),
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )"""),
    (3, "comments and events", """
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            author_id INTEGER NOT NULL REFERENCES users (id),
            body TEXT NOT NULL,
            internal INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        );
        CREATE TABLE events (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            actor_id INTEGER NOT NULL REFERENCES users (id),
            kind TEXT NOT NULL,
            detail TEXT NOT NULL,
            at TEXT NOT NULL
        )"""),
    (4, "indexes", """
        CREATE INDEX idx_tickets_status ON tickets (status, id);
        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);
        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),
]

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def migrate(conn, migrations=MIGRATIONS):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fresh_and_again():\n    \"everything once, then nothing\"\n    conn = connect()\n    assert migrate(conn) == [1, 2, 3, 4] and migrate(conn) == []\n    tables = {r[0] for r in conn.execute(\"SELECT name FROM sqlite_master WHERE type = 'table'\")}\n    assert {\"users\", \"tickets\", \"comments\", \"events\", \"schema_migrations\"} <= tables\n    assert [tuple(r) for r in conn.execute(\"SELECT version, name FROM schema_migrations ORDER BY version\")][-1] == (4, \"indexes\")\n\ndef test_partial_and_new():\n    \"an older database catches up\"\n    conn = connect()\n    assert migrate(conn, MIGRATIONS[:2]) == [1, 2]\n    assert migrate(conn) == [3, 4]\n    extra = MIGRATIONS + [(5, \"ticket source\", \"ALTER TABLE tickets ADD COLUMN source TEXT NOT NULL DEFAULT 'web'\")]\n    assert migrate(conn, extra) == [5]\n    assert \"source\" in [r[1] for r in conn.execute(\"PRAGMA table_info(tickets)\")]\n    try:\n        migrate(conn, MIGRATIONS)\n    except RuntimeError as e:\n        assert str(e) == \"the database is at version 5, newer than this code\"\n    else:\n        raise AssertionError(\"old code must not run on a newer schema\")\n\ndef test_failure_rolls_back():\n    \"a broken migration leaves nothing behind\"\n    conn = connect()\n    migrate(conn)\n    broken = MIGRATIONS + [(5, \"half-broken\", \"CREATE TABLE tags (id INTEGER PRIMARY KEY, name TEXT); CREATE TABLE tags (id INTEGER)\")]\n    try:\n        migrate(conn, broken)\n    except Exception:\n        pass\n    else:\n        raise AssertionError(\"the second CREATE TABLE should fail\")\n    assert \"tags\" not in {r[0] for r in conn.execute(\"SELECT name FROM sqlite_master\")}, \"roll back the whole migration\"\n    assert conn.execute(\"SELECT MAX(version) FROM schema_migrations\").fetchone()[0] == 4\n    for bad in ([(1, \"a\", \"SELECT 1\"), (3, \"c\", \"SELECT 1\")], [(2, \"b\", \"SELECT 1\")]):\n        try:\n            migrate(connect(), bad)\n        except ValueError as e:\n            assert str(e) == \"migrations must be numbered 1, 2, 3... with no gaps\"\n        else:\n            raise AssertionError(bad)")

### Exercise: Step 7: users and tickets

Write `create_user(conn, email, password_hash, role, now)`: insert a user (with `created_at` set to `now`) inside `with conn:` and return the new id; a duplicate email (an `sqlite3.IntegrityError` whose message contains `UNIQUE`) raises `ValueError("email already registered")`, while any other integrity error (such as a role the `CHECK` constraint refuses) is re-raised unchanged. `get_user(conn, user_id)`: a dict with `id`, `email`, `role` and `created_at` (never the password), or `None`. `get_user_by_email(conn, email)`: a dict with `id`, `email`, `password` and `role`, or `None`. `get_ticket(conn, ticket_id)`: every column as a dict, or `None`. And `create_ticket(conn, customer_id, clean, now)`: in one transaction, insert the ticket (`title`, `body`, `priority` from the `clean` dict, defaulting to `"normal"`; `created_at` and `updated_at` both `now`) and an event `(ticket_id, customer_id, "created", TITLE, now)` using the given `_event`; return the new ticket from `get_ticket`.

In [ ]:
import sqlite3

MIGRATIONS = [
    (1, "users", """
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            password TEXT NOT NULL,
            role TEXT NOT NULL CHECK (role IN ('customer', 'agent', 'admin')),
            created_at TEXT NOT NULL
        )"""),
    (2, "tickets", """
        CREATE TABLE tickets (
            id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES users (id),
            assignee_id INTEGER REFERENCES users (id),
            title TEXT NOT NULL,
            body TEXT NOT NULL,
            priority TEXT NOT NULL DEFAULT 'normal' CHECK (priority IN ('low', 'normal', 'high', 'urgent')),
            status TEXT NOT NULL DEFAULT 'open'
                CHECK (status IN ('open', 'in_progress', 'waiting', 'resolved', 'closed')),
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )"""),
    (3, "comments and events", """
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            author_id INTEGER NOT NULL REFERENCES users (id),
            body TEXT NOT NULL,
            internal INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        );
        CREATE TABLE events (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            actor_id INTEGER NOT NULL REFERENCES users (id),
            kind TEXT NOT NULL,
            detail TEXT NOT NULL,
            at TEXT NOT NULL
        )"""),
    (4, "indexes", """
        CREATE INDEX idx_tickets_status ON tickets (status, id);
        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);
        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),
]

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def migrate(conn, migrations=MIGRATIONS):
    """Apply the migrations the database hasn't had yet, each in its own transaction. Returns their versions."""
    conn.execute("CREATE TABLE IF NOT EXISTS schema_migrations (version INTEGER PRIMARY KEY, name TEXT NOT NULL)")
    conn.commit()
    versions = [v for v, _, _ in migrations]
    if versions != list(range(1, len(versions) + 1)):
        raise ValueError("migrations must be numbered 1, 2, 3... with no gaps")
    current = conn.execute("SELECT COALESCE(MAX(version), 0) FROM schema_migrations").fetchone()[0]
    if current > len(versions):
        raise RuntimeError(f"the database is at version {current}, newer than this code")
    applied = []
    for version, name, sql in migrations[current:]:
        try:
            conn.execute("BEGIN")
            for statement in sql.split(";"):
                if statement.strip():
                    conn.execute(statement)
            conn.execute("INSERT INTO schema_migrations (version, name) VALUES (?, ?)", (version, name))
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        applied.append(version)
    return applied

def _dict(row):
    return dict(row) if row is not None else None

def _event(conn, ticket_id, actor_id, kind, detail, now):
    conn.execute("INSERT INTO events (ticket_id, actor_id, kind, detail, at) VALUES (?, ?, ?, ?, ?)",
                 (ticket_id, actor_id, kind, detail, now))


def create_user(conn, email, password_hash, role, now):
    ___


def get_user(conn, user_id):
    ___


def get_user_by_email(conn, email):
    ___


def get_ticket(conn, ticket_id):
    ___


def create_ticket(conn, customer_id, clean, now):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _db():\n    conn = connect()\n    migrate(conn)\n    return conn\n\ndef test_users():\n    \"users, without passwords leaking\"\n    conn = _db()\n    ada = create_user(conn, \"ada@example.org\", \"pbkdf2_sha256$...\", \"customer\", \"2026-11-10T09:00:00Z\")\n    assert ada == 1 and get_user(conn, 1) == {\"id\": 1, \"email\": \"ada@example.org\", \"role\": \"customer\", \"created_at\": \"2026-11-10T09:00:00Z\"}\n    assert get_user_by_email(conn, \"ada@example.org\")[\"password\"] == \"pbkdf2_sha256$...\" and get_user_by_email(conn, \"x@y.z\") is None\n    assert get_user(conn, 99) is None\n    try:\n        create_user(conn, \"ada@example.org\", \"h\", \"customer\", \"t\")\n    except ValueError as e:\n        assert str(e) == \"email already registered\"\n    else:\n        raise AssertionError(\"duplicate\")\n    try:\n        create_user(conn, \"eve@example.org\", \"h\", \"superuser\", \"t\")\n    except ValueError:\n        raise AssertionError(\"a bad role is a programming error, not a duplicate\")\n    except sqlite3.IntegrityError:\n        pass\n\ndef test_tickets():\n    \"a ticket and its event, together\"\n    conn = _db()\n    ada = create_user(conn, \"ada@example.org\", \"h\", \"customer\", \"t0\")\n    t = create_ticket(conn, ada, {\"title\": \"Printer on fire\", \"body\": \"Smoke\", \"priority\": \"urgent\"}, \"2026-11-10T09:01:00Z\")\n    assert t == {\"id\": 1, \"customer_id\": 1, \"assignee_id\": None, \"title\": \"Printer on fire\", \"body\": \"Smoke\", \"priority\": \"urgent\",\n                 \"status\": \"open\", \"created_at\": \"2026-11-10T09:01:00Z\", \"updated_at\": \"2026-11-10T09:01:00Z\"}\n    assert create_ticket(conn, ada, {\"title\": \"Mouse\", \"body\": \"Gone\"}, \"t2\")[\"priority\"] == \"normal\"\n    assert [tuple(r) for r in conn.execute(\"SELECT ticket_id, actor_id, kind, detail FROM events\")] == [(1, 1, \"created\", \"Printer on fire\"), (2, 1, \"created\", \"Mouse\")]\n    try:\n        create_ticket(conn, 42, {\"title\": \"Nobody\", \"body\": \"x\"}, \"t3\")\n    except sqlite3.IntegrityError:\n        pass\n    else:\n        raise AssertionError(\"foreign keys must stop a ticket for a missing user\")\n    assert conn.execute(\"SELECT count(*) FROM events\").fetchone()[0] == 2 and get_ticket(conn, 3) is None")

### Exercise: Step 8: listing tickets

Write `list_tickets(conn, customer_id=None, status=None, priority=None, assignee_id=None, q=None, after_id=0, limit=20)`: tickets (as dicts) with `id > after_id` that match every filter given (equality for the first four; `q` matches titles containing it, using `LIKE ? ESCAPE '\'` with `%`, `_` and `\` escaped by the given `_like`), ordered by id, at most `limit`. Return `(items, next_after)`, where `next_after` is the last item's id if there are more matching tickets after this page, else `None`. Build the `WHERE` clause only from fixed strings; all values are parameters.

In [ ]:
import sqlite3

MIGRATIONS = [
    (1, "users", """
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            password TEXT NOT NULL,
            role TEXT NOT NULL CHECK (role IN ('customer', 'agent', 'admin')),
            created_at TEXT NOT NULL
        )"""),
    (2, "tickets", """
        CREATE TABLE tickets (
            id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES users (id),
            assignee_id INTEGER REFERENCES users (id),
            title TEXT NOT NULL,
            body TEXT NOT NULL,
            priority TEXT NOT NULL DEFAULT 'normal' CHECK (priority IN ('low', 'normal', 'high', 'urgent')),
            status TEXT NOT NULL DEFAULT 'open'
                CHECK (status IN ('open', 'in_progress', 'waiting', 'resolved', 'closed')),
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )"""),
    (3, "comments and events", """
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            author_id INTEGER NOT NULL REFERENCES users (id),
            body TEXT NOT NULL,
            internal INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        );
        CREATE TABLE events (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            actor_id INTEGER NOT NULL REFERENCES users (id),
            kind TEXT NOT NULL,
            detail TEXT NOT NULL,
            at TEXT NOT NULL
        )"""),
    (4, "indexes", """
        CREATE INDEX idx_tickets_status ON tickets (status, id);
        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);
        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),
]

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def migrate(conn, migrations=MIGRATIONS):
    """Apply the migrations the database hasn't had yet, each in its own transaction. Returns their versions."""
    conn.execute("CREATE TABLE IF NOT EXISTS schema_migrations (version INTEGER PRIMARY KEY, name TEXT NOT NULL)")
    conn.commit()
    versions = [v for v, _, _ in migrations]
    if versions != list(range(1, len(versions) + 1)):
        raise ValueError("migrations must be numbered 1, 2, 3... with no gaps")
    current = conn.execute("SELECT COALESCE(MAX(version), 0) FROM schema_migrations").fetchone()[0]
    if current > len(versions):
        raise RuntimeError(f"the database is at version {current}, newer than this code")
    applied = []
    for version, name, sql in migrations[current:]:
        try:
            conn.execute("BEGIN")
            for statement in sql.split(";"):
                if statement.strip():
                    conn.execute(statement)
            conn.execute("INSERT INTO schema_migrations (version, name) VALUES (?, ?)", (version, name))
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        applied.append(version)
    return applied

def _dict(row):
    return dict(row) if row is not None else None

def _event(conn, ticket_id, actor_id, kind, detail, now):
    conn.execute("INSERT INTO events (ticket_id, actor_id, kind, detail, at) VALUES (?, ?, ?, ?, ?)",
                 (ticket_id, actor_id, kind, detail, now))

def create_user(conn, email, password_hash, role, now):
    try:
        with conn:
            cur = conn.execute("INSERT INTO users (email, password, role, created_at) VALUES (?, ?, ?, ?)",
                               (email, password_hash, role, now))
    except sqlite3.IntegrityError as e:
        if "UNIQUE" not in str(e):
            raise
        raise ValueError("email already registered") from None
    return cur.lastrowid

def get_user(conn, user_id):
    return _dict(conn.execute("SELECT id, email, role, created_at FROM users WHERE id = ?", (user_id,)).fetchone())

def create_ticket(conn, customer_id, clean, now):
    with conn:
        cur = conn.execute(
            "INSERT INTO tickets (customer_id, title, body, priority, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?)",
            (customer_id, clean["title"], clean["body"], clean.get("priority", "normal"), now, now))
        _event(conn, cur.lastrowid, customer_id, "created", clean["title"], now)
    return get_ticket(conn, cur.lastrowid)

def get_ticket(conn, ticket_id):
    return _dict(conn.execute("SELECT * FROM tickets WHERE id = ?", (ticket_id,)).fetchone())

def _like(text):
    return "%" + text.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_") + "%"


def list_tickets(conn, customer_id=None, status=None, priority=None, assignee_id=None, q=None, after_id=0, limit=20):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _db():\n    conn = connect()\n    migrate(conn)\n    ada = create_user(conn, \"ada@example.org\", \"h\", \"customer\", \"t\")\n    bob = create_user(conn, \"bob@example.org\", \"h\", \"customer\", \"t\")\n    for i, (who, title, prio) in enumerate([(ada, \"Printer on fire\", \"urgent\"), (bob, \"100% CPU\", \"high\"), (ada, \"Mouse gone\", \"low\"),\n                                           (bob, \"Printer jam\", \"normal\"), (ada, \"Email_bounces\", \"normal\")]):\n        create_ticket(conn, who, {\"title\": title, \"body\": \"x\", \"priority\": prio}, f\"t{i}\")\n    conn.execute(\"UPDATE tickets SET status = 'closed' WHERE id = 3\")\n    conn.commit()\n    return conn\n\ndef test_filters():\n    \"filters combine\"\n    conn = _db()\n    ids = lambda **kw: [t[\"id\"] for t in list_tickets(conn, **kw)[0]]\n    assert ids() == [1, 2, 3, 4, 5] and ids(customer_id=1) == [1, 3, 5] and ids(status=\"open\", customer_id=1) == [1, 5]\n    assert ids(priority=\"normal\") == [4, 5] and ids(q=\"printer\") == [1, 4] and ids(q=\"%\") == [2] and ids(q=\"r_j\") == [] and ids(q=\"l_b\") == [5]\n    assert ids(q=\"_\") == [5] and ids(customer_id=2, q=\"printer\") == [4] and ids(assignee_id=7) == []\n    assert ids(status=\"open' OR '1'='1\") == []\n\ndef test_pages():\n    \"cursor pagination\"\n    conn = _db()\n    page, after = list_tickets(conn, limit=2)\n    assert [t[\"id\"] for t in page] == [1, 2] and after == 2\n    page, after = list_tickets(conn, after_id=after, limit=2)\n    assert [t[\"id\"] for t in page] == [3, 4] and after == 4\n    page, after = list_tickets(conn, after_id=after, limit=2)\n    assert [t[\"id\"] for t in page] == [5] and after is None\n    assert list_tickets(conn, customer_id=1, limit=3) == (list_tickets(conn, customer_id=1)[0], None)\n    assert list_tickets(conn, after_id=5) == ([], None)")

### Exercise: Step 9: changes that can't half-happen

Write `change_status(conn, ticket_id, expected, new, actor_id, now)`: in one transaction, set the ticket's status to `new` and `updated_at` to `now`, **only if** its status is still `expected`; if no row was updated, raise `ConflictError(f"ticket N is no longer EXPECTED")` (which also rolls the transaction back); otherwise add the event `(ticket_id, actor_id, "status", "EXPECTED->NEW", now)` and return the ticket. `assign(conn, ticket_id, agent_id, actor_id, now)`: raise `ValueError(f"user N is not an agent")` unless that user exists with role `agent` or `admin`; then set `assignee_id` and `updated_at`, add the event `"assigned"` with detail `str(agent_id)`, and return the ticket. `add_comment(conn, ticket_id, author_id, body, internal, now)`: insert the comment (`internal` stored as `0` or `1`) and update the ticket's `updated_at`, together; return the comment as a dict. `list_comments(conn, ticket_id, include_internal)`: the ticket's comments by id, leaving out internal ones unless `include_internal`.

In [ ]:
import sqlite3

MIGRATIONS = [
    (1, "users", """
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            password TEXT NOT NULL,
            role TEXT NOT NULL CHECK (role IN ('customer', 'agent', 'admin')),
            created_at TEXT NOT NULL
        )"""),
    (2, "tickets", """
        CREATE TABLE tickets (
            id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES users (id),
            assignee_id INTEGER REFERENCES users (id),
            title TEXT NOT NULL,
            body TEXT NOT NULL,
            priority TEXT NOT NULL DEFAULT 'normal' CHECK (priority IN ('low', 'normal', 'high', 'urgent')),
            status TEXT NOT NULL DEFAULT 'open'
                CHECK (status IN ('open', 'in_progress', 'waiting', 'resolved', 'closed')),
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )"""),
    (3, "comments and events", """
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            author_id INTEGER NOT NULL REFERENCES users (id),
            body TEXT NOT NULL,
            internal INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        );
        CREATE TABLE events (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            actor_id INTEGER NOT NULL REFERENCES users (id),
            kind TEXT NOT NULL,
            detail TEXT NOT NULL,
            at TEXT NOT NULL
        )"""),
    (4, "indexes", """
        CREATE INDEX idx_tickets_status ON tickets (status, id);
        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);
        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),
]

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def migrate(conn, migrations=MIGRATIONS):
    """Apply the migrations the database hasn't had yet, each in its own transaction. Returns their versions."""
    conn.execute("CREATE TABLE IF NOT EXISTS schema_migrations (version INTEGER PRIMARY KEY, name TEXT NOT NULL)")
    conn.commit()
    versions = [v for v, _, _ in migrations]
    if versions != list(range(1, len(versions) + 1)):
        raise ValueError("migrations must be numbered 1, 2, 3... with no gaps")
    current = conn.execute("SELECT COALESCE(MAX(version), 0) FROM schema_migrations").fetchone()[0]
    if current > len(versions):
        raise RuntimeError(f"the database is at version {current}, newer than this code")
    applied = []
    for version, name, sql in migrations[current:]:
        try:
            conn.execute("BEGIN")
            for statement in sql.split(";"):
                if statement.strip():
                    conn.execute(statement)
            conn.execute("INSERT INTO schema_migrations (version, name) VALUES (?, ?)", (version, name))
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        applied.append(version)
    return applied

def _dict(row):
    return dict(row) if row is not None else None

def _event(conn, ticket_id, actor_id, kind, detail, now):
    conn.execute("INSERT INTO events (ticket_id, actor_id, kind, detail, at) VALUES (?, ?, ?, ?, ?)",
                 (ticket_id, actor_id, kind, detail, now))

def create_user(conn, email, password_hash, role, now):
    try:
        with conn:
            cur = conn.execute("INSERT INTO users (email, password, role, created_at) VALUES (?, ?, ?, ?)",
                               (email, password_hash, role, now))
    except sqlite3.IntegrityError as e:
        if "UNIQUE" not in str(e):
            raise
        raise ValueError("email already registered") from None
    return cur.lastrowid

def get_user(conn, user_id):
    return _dict(conn.execute("SELECT id, email, role, created_at FROM users WHERE id = ?", (user_id,)).fetchone())

def create_ticket(conn, customer_id, clean, now):
    with conn:
        cur = conn.execute(
            "INSERT INTO tickets (customer_id, title, body, priority, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?)",
            (customer_id, clean["title"], clean["body"], clean.get("priority", "normal"), now, now))
        _event(conn, cur.lastrowid, customer_id, "created", clean["title"], now)
    return get_ticket(conn, cur.lastrowid)

def get_ticket(conn, ticket_id):
    return _dict(conn.execute("SELECT * FROM tickets WHERE id = ?", (ticket_id,)).fetchone())

class ConflictError(Exception):
    """The row changed since it was read."""


def change_status(conn, ticket_id, expected, new, actor_id, now):
    ___


def assign(conn, ticket_id, agent_id, actor_id, now):
    ___


def add_comment(conn, ticket_id, author_id, body, internal, now):
    ___


def list_comments(conn, ticket_id, include_internal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _db():\n    conn = connect()\n    migrate(conn)\n    create_user(conn, \"ada@example.org\", \"h\", \"customer\", \"t\")\n    create_user(conn, \"grace@example.org\", \"h\", \"agent\", \"t\")\n    create_ticket(conn, 1, {\"title\": \"Printer on fire\", \"body\": \"x\"}, \"t0\")\n    return conn\n\ndef _events(conn):\n    return [tuple(r) for r in conn.execute(\"SELECT kind, detail, actor_id FROM events WHERE kind != 'created' ORDER BY id\")]\n\ndef test_status():\n    \"conditional updates stop lost updates\"\n    conn = _db()\n    assert change_status(conn, 1, \"open\", \"in_progress\", 2, \"t1\")[\"status\"] == \"in_progress\"\n    assert change_status(conn, 1, \"in_progress\", \"resolved\", 2, \"t2\")[\"updated_at\"] == \"t2\"\n    try:\n        change_status(conn, 1, \"in_progress\", \"waiting\", 2, \"t3\")\n    except ConflictError as e:\n        assert str(e) == \"ticket 1 is no longer in_progress\"\n    else:\n        raise AssertionError(\"a stale status must be refused\")\n    assert get_ticket(conn, 1)[\"status\"] == \"resolved\" and _events(conn) == [(\"status\", \"open->in_progress\", 2), (\"status\", \"in_progress->resolved\", 2)]\n\ndef test_assign_and_comments():\n    \"staff only; comments with internal notes\"\n    conn = _db()\n    assert assign(conn, 1, 2, 2, \"t1\")[\"assignee_id\"] == 2\n    for bad in (1, 99):\n        try:\n            assign(conn, 1, bad, 2, \"t2\")\n        except ValueError as e:\n            assert str(e) == f\"user {bad} is not an agent\"\n        else:\n            raise AssertionError(bad)\n    c = add_comment(conn, 1, 2, \"On my way\", False, \"t3\")\n    assert c == {\"id\": 1, \"ticket_id\": 1, \"author_id\": 2, \"body\": \"On my way\", \"internal\": 0, \"created_at\": \"t3\"}\n    add_comment(conn, 1, 2, \"Customer's printer is ancient\", True, \"t4\")\n    assert [x[\"body\"] for x in list_comments(conn, 1, False)] == [\"On my way\"] and len(list_comments(conn, 1, True)) == 2\n    assert get_ticket(conn, 1)[\"updated_at\"] == \"t4\" and _events(conn) == [(\"assigned\", \"2\", 2)]")

### Exercise: Step 10: indexes and statistics

Write `uses_index(conn, sql, params, index)`: `True` if any row of `EXPLAIN QUERY PLAN` for that query (with those parameters) has `INDEX NAME` in its last column (the description). Then `stats(conn)`: a dict with `"by_status"` (every status in `STATUSES` order with its count, `0` if none), `"open_by_priority"` (every priority with its count among tickets that aren't closed) and `"unassigned"` (tickets with no assignee that aren't closed), using `GROUP BY` queries.

In [ ]:
import sqlite3

STATUSES = ("open", "in_progress", "waiting", "resolved", "closed")
PRIORITIES = ("low", "normal", "high", "urgent")

MIGRATIONS = [
    (1, "users", """
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            password TEXT NOT NULL,
            role TEXT NOT NULL CHECK (role IN ('customer', 'agent', 'admin')),
            created_at TEXT NOT NULL
        )"""),
    (2, "tickets", """
        CREATE TABLE tickets (
            id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES users (id),
            assignee_id INTEGER REFERENCES users (id),
            title TEXT NOT NULL,
            body TEXT NOT NULL,
            priority TEXT NOT NULL DEFAULT 'normal' CHECK (priority IN ('low', 'normal', 'high', 'urgent')),
            status TEXT NOT NULL DEFAULT 'open'
                CHECK (status IN ('open', 'in_progress', 'waiting', 'resolved', 'closed')),
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )"""),
    (3, "comments and events", """
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            author_id INTEGER NOT NULL REFERENCES users (id),
            body TEXT NOT NULL,
            internal INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        );
        CREATE TABLE events (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            actor_id INTEGER NOT NULL REFERENCES users (id),
            kind TEXT NOT NULL,
            detail TEXT NOT NULL,
            at TEXT NOT NULL
        )"""),
    (4, "indexes", """
        CREATE INDEX idx_tickets_status ON tickets (status, id);
        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);
        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),
]

def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def migrate(conn, migrations=MIGRATIONS):
    """Apply the migrations the database hasn't had yet, each in its own transaction. Returns their versions."""
    conn.execute("CREATE TABLE IF NOT EXISTS schema_migrations (version INTEGER PRIMARY KEY, name TEXT NOT NULL)")
    conn.commit()
    versions = [v for v, _, _ in migrations]
    if versions != list(range(1, len(versions) + 1)):
        raise ValueError("migrations must be numbered 1, 2, 3... with no gaps")
    current = conn.execute("SELECT COALESCE(MAX(version), 0) FROM schema_migrations").fetchone()[0]
    if current > len(versions):
        raise RuntimeError(f"the database is at version {current}, newer than this code")
    applied = []
    for version, name, sql in migrations[current:]:
        try:
            conn.execute("BEGIN")
            for statement in sql.split(";"):
                if statement.strip():
                    conn.execute(statement)
            conn.execute("INSERT INTO schema_migrations (version, name) VALUES (?, ?)", (version, name))
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        applied.append(version)
    return applied

def _dict(row):
    return dict(row) if row is not None else None

def _event(conn, ticket_id, actor_id, kind, detail, now):
    conn.execute("INSERT INTO events (ticket_id, actor_id, kind, detail, at) VALUES (?, ?, ?, ?, ?)",
                 (ticket_id, actor_id, kind, detail, now))

def create_user(conn, email, password_hash, role, now):
    try:
        with conn:
            cur = conn.execute("INSERT INTO users (email, password, role, created_at) VALUES (?, ?, ?, ?)",
                               (email, password_hash, role, now))
    except sqlite3.IntegrityError as e:
        if "UNIQUE" not in str(e):
            raise
        raise ValueError("email already registered") from None
    return cur.lastrowid

def create_ticket(conn, customer_id, clean, now):
    with conn:
        cur = conn.execute(
            "INSERT INTO tickets (customer_id, title, body, priority, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?)",
            (customer_id, clean["title"], clean["body"], clean.get("priority", "normal"), now, now))
        _event(conn, cur.lastrowid, customer_id, "created", clean["title"], now)
    return get_ticket(conn, cur.lastrowid)

def get_ticket(conn, ticket_id):
    return _dict(conn.execute("SELECT * FROM tickets WHERE id = ?", (ticket_id,)).fetchone())


def uses_index(conn, sql, params, index):
    ___


def stats(conn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _db():\n    conn = connect()\n    migrate(conn)\n    create_user(conn, \"ada@example.org\", \"h\", \"customer\", \"t\")\n    for title, prio in [(\"A\", \"urgent\"), (\"B\", \"low\"), (\"C\", \"low\"), (\"D\", \"normal\")]:\n        create_ticket(conn, 1, {\"title\": title * 3, \"body\": \"x\", \"priority\": prio}, \"t\")\n    conn.execute(\"UPDATE tickets SET status = 'closed' WHERE id = 2\")\n    conn.execute(\"UPDATE tickets SET status = 'in_progress', assignee_id = 1 WHERE id = 1\")\n    conn.commit()\n    return conn\n\ndef test_indexes():\n    \"the plan uses the indexes\"\n    conn = _db()\n    assert uses_index(conn, \"SELECT * FROM tickets WHERE status = ? AND id > ? ORDER BY id LIMIT 20\", (\"open\", 0), \"idx_tickets_status\")\n    assert uses_index(conn, \"SELECT * FROM tickets WHERE customer_id = ? ORDER BY id\", (1,), \"idx_tickets_customer\")\n    assert not uses_index(conn, \"SELECT * FROM tickets WHERE title = ?\", (\"x\",), \"idx_tickets_status\")\n    assert uses_index(conn, \"SELECT * FROM comments WHERE ticket_id = ? ORDER BY id\", (1,), \"idx_comments_ticket\")\n\ndef test_stats():\n    \"every status, even at zero\"\n    conn = _db()\n    assert stats(conn) == {\"by_status\": {\"open\": 2, \"in_progress\": 1, \"waiting\": 0, \"resolved\": 0, \"closed\": 1},\n                           \"open_by_priority\": {\"low\": 1, \"normal\": 1, \"high\": 0, \"urgent\": 1}, \"unassigned\": 2}\n    empty = connect()\n    migrate(empty)\n    assert stats(empty)[\"by_status\"] == {s: 0 for s in STATUSES} and stats(empty)[\"unassigned\"] == 0\n    assert list(stats(conn)[\"by_status\"]) == list(STATUSES)")

In [ ]:
"""The helpdesk's database: migrations and the queries the app needs (Week 80, Day 2).

Every query uses parameters; every change that touches more than one row runs in a transaction.
"""
import sqlite3

MIGRATIONS = [
    (1, "users", """
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            password TEXT NOT NULL,
            role TEXT NOT NULL CHECK (role IN ('customer', 'agent', 'admin')),
            created_at TEXT NOT NULL
        )"""),
    (2, "tickets", """
        CREATE TABLE tickets (
            id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES users (id),
            assignee_id INTEGER REFERENCES users (id),
            title TEXT NOT NULL,
            body TEXT NOT NULL,
            priority TEXT NOT NULL DEFAULT 'normal' CHECK (priority IN ('low', 'normal', 'high', 'urgent')),
            status TEXT NOT NULL DEFAULT 'open'
                CHECK (status IN ('open', 'in_progress', 'waiting', 'resolved', 'closed')),
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )"""),
    (3, "comments and events", """
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            author_id INTEGER NOT NULL REFERENCES users (id),
            body TEXT NOT NULL,
            internal INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        );
        CREATE TABLE events (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            actor_id INTEGER NOT NULL REFERENCES users (id),
            kind TEXT NOT NULL,
            detail TEXT NOT NULL,
            at TEXT NOT NULL
        )"""),
    (4, "indexes", """
        CREATE INDEX idx_tickets_status ON tickets (status, id);
        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);
        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),
]


class ConflictError(Exception):
    """The row changed since it was read."""


def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def migrate(conn, migrations=MIGRATIONS):
    """Apply the migrations the database hasn't had yet, each in its own transaction. Returns their versions."""
    conn.execute("CREATE TABLE IF NOT EXISTS schema_migrations (version INTEGER PRIMARY KEY, name TEXT NOT NULL)")
    conn.commit()
    versions = [v for v, _, _ in migrations]
    if versions != list(range(1, len(versions) + 1)):
        raise ValueError("migrations must be numbered 1, 2, 3... with no gaps")
    current = conn.execute("SELECT COALESCE(MAX(version), 0) FROM schema_migrations").fetchone()[0]
    if current > len(versions):
        raise RuntimeError(f"the database is at version {current}, newer than this code")
    applied = []
    for version, name, sql in migrations[current:]:
        try:
            conn.execute("BEGIN")
            for statement in sql.split(";"):
                if statement.strip():
                    conn.execute(statement)
            conn.execute("INSERT INTO schema_migrations (version, name) VALUES (?, ?)", (version, name))
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        applied.append(version)
    return applied


def _dict(row):
    return dict(row) if row is not None else None


def _event(conn, ticket_id, actor_id, kind, detail, now):
    conn.execute("INSERT INTO events (ticket_id, actor_id, kind, detail, at) VALUES (?, ?, ?, ?, ?)",
                 (ticket_id, actor_id, kind, detail, now))


def create_user(conn, email, password_hash, role, now):
    try:
        with conn:
            cur = conn.execute("INSERT INTO users (email, password, role, created_at) VALUES (?, ?, ?, ?)",
                               (email, password_hash, role, now))
    except sqlite3.IntegrityError as e:
        if "UNIQUE" not in str(e):
            raise
        raise ValueError("email already registered") from None
    return cur.lastrowid


def get_user(conn, user_id):
    return _dict(conn.execute("SELECT id, email, role, created_at FROM users WHERE id = ?", (user_id,)).fetchone())


def get_user_by_email(conn, email):
    return _dict(conn.execute("SELECT id, email, password, role FROM users WHERE email = ?", (email,)).fetchone())


def create_ticket(conn, customer_id, clean, now):
    with conn:
        cur = conn.execute(
            "INSERT INTO tickets (customer_id, title, body, priority, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?)",
            (customer_id, clean["title"], clean["body"], clean.get("priority", "normal"), now, now))
        _event(conn, cur.lastrowid, customer_id, "created", clean["title"], now)
    return get_ticket(conn, cur.lastrowid)


def get_ticket(conn, ticket_id):
    return _dict(conn.execute("SELECT * FROM tickets WHERE id = ?", (ticket_id,)).fetchone())


def _like(text):
    return "%" + text.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_") + "%"


def list_tickets(conn, customer_id=None, status=None, priority=None, assignee_id=None, q=None, after_id=0, limit=20):
    """Tickets matching the filters, by id, after a cursor. Returns (items, next_after or None)."""
    where, values = ["id > ?"], [after_id]
    for column, value in (("customer_id", customer_id), ("status", status), ("priority", priority), ("assignee_id", assignee_id)):
        if value is not None:
            where.append(f"{column} = ?")
            values.append(value)
    if q:
        where.append("title LIKE ? ESCAPE '\\'")
        values.append(_like(q))
    rows = conn.execute(f"SELECT * FROM tickets WHERE {' AND '.join(where)} ORDER BY id LIMIT ?", values + [limit + 1]).fetchall()
    items = [dict(r) for r in rows[:limit]]
    return items, (items[-1]["id"] if len(rows) > limit else None)


def update_ticket(conn, ticket_id, changes, actor_id, now):
    if not changes:
        return get_ticket(conn, ticket_id)
    columns = sorted(changes)
    with conn:
        conn.execute(f"UPDATE tickets SET {', '.join(f'{c} = ?' for c in columns)}, updated_at = ? WHERE id = ?",
                     [changes[c] for c in columns] + [now, ticket_id])
        _event(conn, ticket_id, actor_id, "edited", ",".join(columns), now)
    return get_ticket(conn, ticket_id)


def change_status(conn, ticket_id, expected, new, actor_id, now):
    """Move a ticket from `expected` to `new`, unless someone changed its status first."""
    with conn:
        cur = conn.execute("UPDATE tickets SET status = ?, updated_at = ? WHERE id = ? AND status = ?",
                           (new, now, ticket_id, expected))
        if cur.rowcount != 1:
            raise ConflictError(f"ticket {ticket_id} is no longer {expected}")
        _event(conn, ticket_id, actor_id, "status", f"{expected}->{new}", now)
    return get_ticket(conn, ticket_id)


def assign(conn, ticket_id, agent_id, actor_id, now):
    agent = get_user(conn, agent_id)
    if agent is None or agent["role"] not in ("agent", "admin"):
        raise ValueError(f"user {agent_id} is not an agent")
    with conn:
        conn.execute("UPDATE tickets SET assignee_id = ?, updated_at = ? WHERE id = ?", (agent_id, now, ticket_id))
        _event(conn, ticket_id, actor_id, "assigned", str(agent_id), now)
    return get_ticket(conn, ticket_id)


def add_comment(conn, ticket_id, author_id, body, internal, now):
    with conn:
        cur = conn.execute("INSERT INTO comments (ticket_id, author_id, body, internal, created_at) VALUES (?, ?, ?, ?, ?)",
                           (ticket_id, author_id, body, int(internal), now))
        conn.execute("UPDATE tickets SET updated_at = ? WHERE id = ?", (now, ticket_id))
    return _dict(conn.execute("SELECT * FROM comments WHERE id = ?", (cur.lastrowid,)).fetchone())


def list_comments(conn, ticket_id, include_internal):
    sql = "SELECT * FROM comments WHERE ticket_id = ?" + ("" if include_internal else " AND internal = 0") + " ORDER BY id"
    return [dict(r) for r in conn.execute(sql, (ticket_id,)).fetchall()]


def delete_ticket(conn, ticket_id):
    with conn:
        return conn.execute("DELETE FROM tickets WHERE id = ?", (ticket_id,)).rowcount == 1


def events(conn, ticket_id):
    return [dict(r) for r in conn.execute("SELECT kind, detail, actor_id, at FROM events WHERE ticket_id = ? ORDER BY id", (ticket_id,))]


def stats(conn):
    by_status = {s: 0 for s in ("open", "in_progress", "waiting", "resolved", "closed")}
    by_status.update(dict(conn.execute("SELECT status, COUNT(*) FROM tickets GROUP BY status").fetchall()))
    by_priority = {p: 0 for p in ("low", "normal", "high", "urgent")}
    by_priority.update(dict(conn.execute("SELECT priority, COUNT(*) FROM tickets WHERE status != 'closed' GROUP BY priority").fetchall()))
    unassigned = conn.execute("SELECT COUNT(*) FROM tickets WHERE assignee_id IS NULL AND status != 'closed'").fetchone()[0]
    return {"by_status": by_status, "open_by_priority": by_priority, "unassigned": unassigned}


conn = connect()
print("applied", migrate(conn))
ada = create_user(conn, "ada@example.org", "hash", "customer", "2026-11-10T09:00:00Z")
grace = create_user(conn, "grace@example.org", "hash", "agent", "2026-11-10T09:00:00Z")
ticket = create_ticket(conn, ada, {"title": "Printer on fire", "body": "Smoke everywhere", "priority": "urgent"}, "2026-11-10T09:01:00Z")
assign(conn, ticket["id"], grace, grace, "2026-11-10T09:02:00Z")
change_status(conn, ticket["id"], "open", "in_progress", grace, "2026-11-10T09:02:30Z")
add_comment(conn, ticket["id"], grace, "Unplug it, please", False, "2026-11-10T09:03:00Z")
try:
    change_status(conn, ticket["id"], "open", "closed", ada, "2026-11-10T09:04:00Z")
except ConflictError as e:
    print("refused:", e)
for event in events(conn, ticket["id"]):
    print(event["at"][11:16], event["kind"], event["detail"])
print(stats(conn)["by_status"])

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

Yesterday's decisions are in `helpdesk_domain` (in the files panel). Today's module is `helpdesk_db`: everything that talks to the database, and nothing else. The web layer (tomorrow) will call functions such as `create_ticket(conn, customer_id, clean, now)` and never write SQL itself, so all the SQL lives in one place, where it can be reviewed for injection and tested against a real database.

## The schema

Four tables hold everything:

```sql
users    (id, email UNIQUE, password, role CHECK (role IN (...)), created_at)
tickets  (id, customer_id -> users, assignee_id -> users, title, body,
          priority CHECK (...), status CHECK (...), created_at, updated_at)
comments (id, ticket_id -> tickets ON DELETE CASCADE, author_id -> users, body, internal, created_at)
events   (id, ticket_id -> tickets ON DELETE CASCADE, actor_id -> users, kind, detail, at)
```

Three choices make the **database** enforce rules, so a bug in the code (or a hand-written fix at 2 a.m.) can't store nonsense:

- **Constraints**: `NOT NULL`, `UNIQUE` on emails, and `CHECK` constraints that repeat the domain's lists of roles, priorities and statuses.
- **Foreign keys** (`REFERENCES`): a ticket can't name a customer who doesn't exist. SQLite only enforces them after `PRAGMA foreign_keys = ON` on each connection, which `connect` does. `ON DELETE CASCADE` deletes a ticket's comments and events with it.
- **An audit trail**: every significant change adds a row to `events` (who did what, when), written in the **same transaction** as the change, so the history can't disagree with the data.

Times are stored as ISO 8601 text in UTC (`2026-11-10T09:00:00Z`), which sorts correctly as text. The `connect` helper sets `row_factory = sqlite3.Row`, so rows can be turned into dicts with `dict(row)`.

```python
def connect(path=":memory:"):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn
```

## Step 6: migrations

A database outlives the code that first created it. When the next version needs a new column or index, you can't drop the tables and start again: the data is there. So the schema is built by **migrations**: numbered changes, applied in order, each exactly once, with a table in the database recording which have run. Deploying a new version runs the ones that haven't (Week 77 Day 5); a fresh database runs them all.

```python
MIGRATIONS = [
    (1, "users", """
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            password TEXT NOT NULL,
            role TEXT NOT NULL CHECK (role IN ('customer', 'agent', 'admin')),
            created_at TEXT NOT NULL
        )"""),
    (2, "tickets", """
        CREATE TABLE tickets (
            id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES users (id),
            assignee_id INTEGER REFERENCES users (id),
            title TEXT NOT NULL,
            body TEXT NOT NULL,
            priority TEXT NOT NULL DEFAULT 'normal' CHECK (priority IN ('low', 'normal', 'high', 'urgent')),
            status TEXT NOT NULL DEFAULT 'open'
                CHECK (status IN ('open', 'in_progress', 'waiting', 'resolved', 'closed')),
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        )"""),
    (3, "comments and events", """
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            author_id INTEGER NOT NULL REFERENCES users (id),
            body TEXT NOT NULL,
            internal INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL
        );
        CREATE TABLE events (
            id INTEGER PRIMARY KEY,
            ticket_id INTEGER NOT NULL REFERENCES tickets (id) ON DELETE CASCADE,
            actor_id INTEGER NOT NULL REFERENCES users (id),
            kind TEXT NOT NULL,
            detail TEXT NOT NULL,
            at TEXT NOT NULL
        )"""),
    (4, "indexes", """
        CREATE INDEX idx_tickets_status ON tickets (status, id);
        CREATE INDEX idx_tickets_customer ON tickets (customer_id, id);
        CREATE INDEX idx_comments_ticket ON comments (ticket_id, id)"""),
]
```

The rules for a safe migration runner:

- numbers are `1, 2, 3...` with no gaps or repeats (two people adding "migration 5" on different branches must be caught);
- each migration runs in its **own transaction** with its record in `schema_migrations`, so it's applied completely or not at all (SQLite, unlike some databases, can roll back `CREATE TABLE`);
- a database at a **higher** version than the code knows about means old code is running against a newer schema: refuse to start rather than guess.

In Python's `sqlite3`, run `conn.execute("BEGIN")` to start the transaction explicitly (so the `CREATE` statements are part of it), then `conn.commit()` or, on any error, `conn.rollback()` and re-raise. `executescript` can't be used here, because it commits on its own; split the migration's SQL on `;` instead (fine for migrations you write yourself, which don't contain `;` inside strings).

## Step 7: users and tickets

The basic operations are small functions, each one query (or one transaction), all with parameters. Two details matter:

- **Translate database errors into domain errors.** A duplicate email is an `sqlite3.IntegrityError`, but the web layer shouldn't need to know that; `create_user` raises `ValueError("email already registered")`, which tomorrow becomes a `409`.
- **Return plain dicts**, not database rows or cursors, so the layers above don't depend on the database library either.

Creating a ticket writes two rows, the ticket and its `created` event, in one transaction. In Python's `sqlite3`, **`with conn:`** commits the transaction when the block finishes, or rolls it back if it raises.

## Step 8: listing tickets

The ticket list is the API's busiest query, with optional filters (status, priority, assignee, a search in the title, and, for customers, only their own tickets) and **cursor pagination** (Week 77): instead of "page 3", the client sends the id of the last ticket it has seen, and gets the next ones after it. Cursors don't skip or repeat items when tickets are added meanwhile, and they stay fast on page 1000, because `WHERE id > ? ORDER BY id LIMIT ?` walks an index instead of counting past everything before.

Building a query with optional filters is where injection creeps in, so the rule from Week 79 applies exactly: the SQL text is built only from **fixed pieces written in the code** (`"status = ?"`), and every value is a parameter. To know whether there's a next page without a second query, ask for **one more** row than the limit: if it comes back, there's more, and the cursor is the last id on this page.

## Step 9: changes that can't half-happen

Three operations change tickets, and each must be safe when things go wrong or happen at once:

- **Status changes and lost updates.** Two agents load ticket 7 (status `in_progress`). One marks it `resolved`; a moment later the other, still looking at `in_progress`, marks it `waiting`. A plain `UPDATE ... SET status = 'waiting' WHERE id = 7` silently overwrites the resolution. The fix is a **conditional update**: `UPDATE ... WHERE id = ? AND status = ?` with the status the agent saw. If someone changed it meanwhile, no row matches (`cursor.rowcount == 0`), and the function raises `ConflictError`, which becomes a `409` asking the person to reload. This is Week 77's ETag idea, in the database.
- **Assignment** must check that the assignee is staff, and records an event.
- **Comments** insert the comment and touch the ticket's `updated_at`, in one transaction.

Each writes its change and its `events` row inside one `with conn:` block: if anything fails, neither is kept.

## Step 10: indexes and statistics

Without an index, `WHERE status = 'open' ORDER BY id` reads **every** ticket. Migration 4 adds indexes on exactly the columns the common queries filter by, followed by `id` so the rows come out already in order: `(status, id)`, `(customer_id, id)`, and `(ticket_id, id)` for comments. Don't guess whether an index is used: ask the database. **`EXPLAIN QUERY PLAN`** in front of a query returns rows describing how SQLite will run it, such as `SEARCH tickets USING INDEX idx_tickets_status (status=? AND id>?)`; a `SCAN tickets` means it reads the whole table. A test that checks the plan stops a later change from quietly losing the index.

The statistics the manager asked for are **aggregate** queries (Week 75): `GROUP BY status` with `COUNT(*)`. One detail: a status with no tickets doesn't appear in a `GROUP BY` result at all, so start from a dict with every status at zero and update it, or the dashboard will be missing rows exactly when things are quiet.

## Checkpoint

The whole database layer at work: migrate, sign two people up, open a ticket, work on it, and look at its history, the way tomorrow's endpoints will.

The full `helpdesk_db` module, with `update_ticket`, `delete_ticket` and `events` as well, is in tomorrow's files panel.

## Quiz

## Before you go

Think of one more thing the helpdesk might need to store later (tags on tickets, say) and write the migration for it as migration 5. Tomorrow: the **endpoints and authentication**, where the domain rules and the database meet HTTP.